<a href="https://colab.research.google.com/github/yamak493/nlf/blob/claude/kind-mendel-tdveor/mp4_to_mannequin_ja.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🕺 mp4 → モーション抽出 → 踊るマネキン動画（音声付き）

このノートブックは、**アップロードした mp4 の「始点秒〜終点秒」の区間から人物の 3D モーションを抽出し、
そのモーションで動くマネキンの動画（元動画の音声付き mp4）を書き出す**ためのものです。

使用する技術:

| 役割 | 使うもの |
|---|---|
| 画像 → 3D 人体（頂点・関節・SMPL パラメータ） | [NLF (Neural Localizer Fields)](https://github.com/isarandi/nlf) の TorchScript モデル `v0.3.2` |
| 3D 頂点 → SMPL パラメータへのフィッティング | [SMPLFitter](https://github.com/isarandi/smplfitter)（NLF の内部でも使われています） |
| 動画の切り出し・音声の合成 | ffmpeg（`imageio-ffmpeg` に同梱のバイナリを使うので別途インストール不要） |

## 処理の流れ

1. ライブラリを自動インストール（`smplfitter` など）
2. NLF の学習済みモデル（約 470&nbsp;MB）を自動ダウンロード
3. mp4 をファイル選択でアップロード
4. 始点秒・終点秒を指定して、その区間を切り出し（映像と音声）
5. 1 フレームずつ NLF で推論 → 人物を 1 人選んで追跡 → **モーションデータ**（SMPL の pose / betas / trans / 関節 / 頂点）を取得
6. 時間方向に平滑化して `motion.npz` に保存
7. モーションを反映した**マネキンのメッシュ**を作ってレンダリング
8. 元動画の音声を合成して `mannequin_with_audio.mp4` を出力・再生・ダウンロード

## 実行前の注意

* **GPU ランタイムが必須**です。Colab では「ランタイム → ランタイムのタイプを変更 → ハードウェア アクセラレータ: GPU」を選んでください（NLF は半精度で動くため CPU では実行できません）。
* 処理時間の目安（Colab T4 / 720p）: **推論 約 0.3〜0.6 秒/フレーム**。まずは **5〜10 秒程度**の区間で試してください。
* マネキンの見た目は 2 種類あります。
  * **パーツ凸包マネキン（既定）**: SMPL の公式ファイルが無くても動きます。木製デッサン人形のような見た目になります。
  * **SMPL メッシュ**: SMPL 公式配布ファイル（要ユーザー登録）がある場合のみ。人体そのままの滑らかなメッシュになります。
* ライセンス: NLF のモデルは**非商用の研究用途**で公開されています。SMPL 系ボディモデルは [smpl.is.tue.mpg.de](https://smpl.is.tue.mpg.de/) 等での登録・ライセンス同意が必要です。アップロードする動画は自分に権利があるものを使ってください。

---
## 1. ライブラリのインストール

必要なパッケージを自動で入れます（Colab では PyTorch は既に入っているのでそのまま使います）。
初回は 1〜2 分ほどかかります。

In [ ]:
#@title 1. セットアップ（実行するだけ） { display-mode: "form" }
import importlib.util
import os
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
print('Colab 上で実行中:', IN_COLAB)


def pip_install(*pkgs):
    print('インストール中:', ' '.join(pkgs))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)


# --- PyTorch（Colab には最初から入っています） ---
try:
    import torch
    import torchvision  # NLF の TorchScript を読むために必須
except ImportError:
    pip_install('torch', 'torchvision')
    import torch
    import torchvision

# --- その他の依存パッケージ ---
required = [
    ('smplfitter', 'smplfitter'),   # SMPL フィッティング（NLF 内部でも使用）
    ('scipy', 'scipy'),
    ('trimesh', 'trimesh'),
    ('imageio', 'imageio'),
    ('imageio_ffmpeg', 'imageio-ffmpeg'),  # ffmpeg バイナリ同梱
    ('matplotlib', 'matplotlib'),
    ('tqdm', 'tqdm'),
    ('PIL', 'pillow'),
]
missing = [pkg for mod, pkg in required if importlib.util.find_spec(mod) is None]
if missing:
    pip_install(*missing)
else:
    print('依存パッケージはすべて揃っています。')

import numpy as np

# numpy 2.x では np.infty が削除されたが、pyrender など一部ライブラリがまだ使うので補う
if not hasattr(np, 'infty'):
    np.infty = np.inf

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print()
print('PyTorch:', torch.__version__, '/ NumPy:', np.__version__)
print('デバイス:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️ GPU が見つかりません。NLF は half 精度で動作するため GPU が必要です。')
    print('   Colab なら「ランタイム → ランタイムのタイプを変更 → GPU」を選んでから、')
    print('   このノートブックを最初から実行し直してください。')

WORK_DIR = os.path.abspath('nlf_mannequin')
os.makedirs(WORK_DIR, exist_ok=True)
print('作業ディレクトリ:', WORK_DIR)

---
## 2. NLF 学習済みモデルのダウンロード

[NLF v0.3.2 リリース](https://github.com/isarandi/nlf/releases/tag/v0.3.2) の
`nlf_l_multi_0.3.2.torchscript`（約 470&nbsp;MB / EfficientNetV2-L バックボーン）を取得します。

このモデルは **人物検出 → 3D 頂点・関節の推定 → SMPL パラメータへのフィッティング（SMPLFitter）** までを
1 つの TorchScript にまとめたものです。一度ダウンロードすれば以降のセル実行では再利用されます。

In [ ]:
#@title 2. NLF モデルの自動ダウンロードと読み込み { display-mode: "form" }
import time
import urllib.request

MODEL_URL = 'https://github.com/isarandi/nlf/releases/download/v0.3.2/nlf_l_multi_0.3.2.torchscript'
MODEL_PATH = os.path.join(WORK_DIR, 'nlf_l_multi_0.3.2.torchscript')
MIN_EXPECTED_BYTES = 300 * 1024 ** 2


def download(url, path, min_bytes=0):
    if os.path.exists(path) and os.path.getsize(path) >= min_bytes:
        print(f'既にダウンロード済み: {path} ({os.path.getsize(path) / 1024 ** 2:.0f} MB)')
        return path
    tmp = path + '.part'
    print('ダウンロード中:', url)
    t0 = time.time()
    with urllib.request.urlopen(url) as resp, open(tmp, 'wb') as f:
        total = int(resp.headers.get('Content-Length', 0))
        done = 0
        while True:
            chunk = resp.read(1024 * 1024)
            if not chunk:
                break
            f.write(chunk)
            done += len(chunk)
            if total:
                bar = '█' * int(30 * done / total)
                print(f'\r  [{bar:<30}] {done / 1024 ** 2:7.0f} / {total / 1024 ** 2:.0f} MB',
                      end='', flush=True)
    print()
    os.replace(tmp, path)
    print(f'完了（{time.time() - t0:.0f} 秒）:', path)
    return path


download(MODEL_URL, MODEL_PATH, MIN_EXPECTED_BYTES)

print('モデルを読み込み中…（30 秒ほどかかります）')
nlf_model = torch.jit.load(MODEL_PATH).to(DEVICE).eval()
print('読み込み完了 ✅')

---
## 3.（任意）SMPL 公式ボディモデルの用意

* **何もしなくても動きます。** その場合は SMPL の頂点を体のパーツごとに凸包（convex hull）で包んだ
  「デッサン人形風マネキン」でレンダリングします（メッシュの面情報が不要な方式です）。
* SMPL の公式ファイルがあると、**人体メッシュそのもの**をマネキンとして描画でき、さらに
  SMPLFitter で「全フレーム共通の体型（betas）」に整えるリフィットも実行できます。

公式ファイルは [smpl.is.tue.mpg.de](https://smpl.is.tue.mpg.de/) でのユーザー登録とライセンス同意が必要です。
登録済みなら、下の `DOWNLOAD_SMPL_MODEL` を `True` にして実行すると、`smplfitter` のダウンローダ経由で
メールアドレスとパスワードを聞かれ、自動で配置されます（入力内容はどこにも保存されません）。
既に手元にファイルがある場合は `body_models/smpl/` 以下に置いてください。

In [ ]:
#@title 3. ボディモデルの検出（任意ダウンロード） { display-mode: "form" }
DOWNLOAD_SMPL_MODEL = False  #@param {type:"boolean"}

BODY_MODELS_DIR = os.path.join(WORK_DIR, 'body_models')
os.makedirs(BODY_MODELS_DIR, exist_ok=True)
os.environ['SMPLFITTER_BODY_MODELS'] = BODY_MODELS_DIR
# 想定する配置: body_models/smpl/basicmodel_neutral_lbs_10_207_0_v1.1.0.pkl

if DOWNLOAD_SMPL_MODEL:
    import getpass
    from pathlib import Path
    from urllib.parse import quote
    try:
        from smplfitter.download import _download_smpl, _make_opener
        email = input('MPI (smpl.is.tue.mpg.de) の登録メールアドレス: ')
        password = getpass.getpass('パスワード: ')
        auth = f'username={quote(email, safe="")}&password={quote(password, safe="")}'.encode()
        _download_smpl(_make_opener(), auth, Path(BODY_MODELS_DIR))
    except Exception as e:
        print('⚠️ ダウンロードに失敗しました:', repr(e))
        print('   登録が済んでいるか、メール/パスワードが正しいかを確認してください。')
        print('   （このまま進めてもパーツ凸包マネキンで動画は作れます）')


def load_body_model(model_name='smpl', gender='neutral'):
    # 公式ボディモデルが見つかればロードする。無ければ None を返す。
    try:
        from smplfitter.pt import BodyModel
        bm = BodyModel(model_name, gender, num_betas=10)
        return bm
    except Exception as e:
        print(f'SMPL 公式ファイルは見つかりませんでした（{type(e).__name__}）。')
        return None


BODY_MODEL = load_body_model('smpl')
SMPL_FACES = None if BODY_MODEL is None else np.asarray(BODY_MODEL.faces, np.int32)
if SMPL_FACES is None:
    print('→ マネキンは「パーツ凸包」方式で作ります（公式ファイル不要）。')
else:
    print(f'→ SMPL メッシュが使えます（面数 {len(SMPL_FACES)}）。')

---
## 4. mp4 ファイルのアップロード（ファイル選択）

* **Colab**: 実行するとファイル選択ダイアログが開きます。手元の mp4 を選んでください。
* **ローカルの Jupyter**: ファイル選択ボタンが表示されます。ファイルを選んだあと、**もう一度このセルを実行**してください。
  （`VIDEO_PATH` に直接パスを書いてもかまいません。）

In [ ]:
#@title 4. 動画をアップロード { display-mode: "form" }
import shutil

VIDEO_PATH = ''  #@param {type:"string"}

if VIDEO_PATH:
    assert os.path.exists(VIDEO_PATH), f'ファイルが見つかりません: {VIDEO_PATH}'
elif IN_COLAB:
    from google.colab import files
    print('mp4 ファイルを選択してください…')
    uploaded = files.upload()
    assert uploaded, 'ファイルが選択されませんでした。'
    name = list(uploaded.keys())[0]
    VIDEO_PATH = os.path.join(WORK_DIR, os.path.basename(name))
    shutil.move(name, VIDEO_PATH)
else:
    import ipywidgets
    from IPython.display import display
    _uploader = globals().get('_uploader') or ipywidgets.FileUpload(accept='.mp4', multiple=False)
    if not _uploader.value:
        display(_uploader)
        raise SystemExit('☝️ ファイルを選んでから、このセルをもう一度実行してください。')
    item = (_uploader.value[0] if isinstance(_uploader.value, (list, tuple))
            else list(_uploader.value.values())[0])
    fname = item.get('name') or item['metadata']['name']
    VIDEO_PATH = os.path.join(WORK_DIR, os.path.basename(fname))
    with open(VIDEO_PATH, 'wb') as f:
        f.write(item['content'])

print('入力動画:', VIDEO_PATH, f'({os.path.getsize(VIDEO_PATH) / 1024 ** 2:.1f} MB)')

---
## 5. 動画の情報を確認して、切り出す区間などを設定

ここで **始点秒数 `START_SEC`・終点秒数 `END_SEC`** を指定します。まずは 5〜10 秒程度で試すのがおすすめです。

### 切り出し・人物

| 設定 | 意味 |
|---|---|
| `START_SEC` / `END_SEC` | 切り出す区間（秒）。`END_SEC = 0` なら動画の最後まで |
| `TARGET_FPS` | 処理する fps（`0` で元動画のまま）。小さくすると速くなります |
| `MAX_HEIGHT` | 推論時に縮小する高さの上限（720 程度が速度と精度のバランス◎） |
| `PERSON_SELECT` | 最初のフレームでどの人物を主役にするか（`largest`=一番大きく写っている人 / `center`=画面中央の人） |

### 推論（GPU の使い方と精度）

| 設定 | 意味 |
|---|---|
| `BATCH_SIZE` | 一度に GPU へ送るフレーム数 |
| `NUM_AUG` | 1 人あたりのテスト時データ拡張（TTA）の枚数。**奇数のみ**（偶数だと拡張の振り方が左右非対称になります） |
| `ANTIALIAS` | クロップを作るときの超サンプリング倍率。小さく写っている人物に効きます（NLF 本体のコメントに「4 は精度が上がることがある」とあります） |
| `DETECTOR_THRESHOLD` | 人物検出のしきい値。検出が途切れるときは 0.15 程度まで下げてください |

NLF が GPU に流すのは「**フレーム数 × 人数 × `NUM_AUG`** 枚のクロップ（384×384）」です。
1 人しか写っていない動画で `BATCH_SIZE=4`・`NUM_AUG=1` だと 4 枚しか流れず、GPU がほとんど遊びます。
**GPU メモリが余っているときは、まず `BATCH_SIZE` を増やし、次に `NUM_AUG` を増やしてください。**

`NUM_AUG` は、明るさ・面内回転・拡大率・左右反転を変えた複数のクロップで推論し、
**不確実性で重み付けした幾何中央値**で統合する仕組みです（NLF 本体の機能）。
つまり「体の細かい震え」の原因であるフレームごとの推定ノイズを、後処理ではなく**発生源で**減らせます。
所要時間はおおよそ `NUM_AUG` に比例します（1 → 3 で約 2〜3 倍）。メモリが足りなければ自動でバッチを分割して再試行します。

### モーションの整形（震え・足滑り対策）

| 設定 | 意味 |
|---|---|
| `MEDIAN_WINDOW` | 単発の外れフレームを除く中央値フィルタの窓（フレーム。1 で無効） |
| `SMOOTH_CUTOFF_HZ` | 手足の回転と全身の位置のローパス遮断周波数。小さいほど滑らか、大きいほどキレが残る |
| `ROOT_CUTOFF_HZ` | **全身の向き**のローパス遮断周波数。胴全体が細かく震えるときはここを下げます |
| `DEPTH_EXTRA_SMOOTH` | 奥行き（z）だけ強めに平滑化。奥行きのブレは減りますが、**前後に動く振付では足が滑りやすくなる**ので既定はオフ（奥行きのドリフトは 8c の接地補正で直します） |
| `FOOT_LOCK` | 接地している足が地面上で止まるよう全身の位置を補正（足滑り対策） |
| `ROOT_MOTION` | `locked`=その場で踊る（低周波のドリフトを除去）/ `smoothed`=移動を残して滑らかに / `full`=補正後の移動をそのまま |

平滑化はゼロ位相フィルタ（前後両方向にかける）なので**動きが遅れません**。

### 出力

| 設定 | 意味 |
|---|---|
| `MANNEQUIN_STYLE` | `auto`（公式 SMPL があればメッシュ、無ければパーツ凸包）/ `parts` / `smpl_mesh` |
| `CAMERA_MODE` | `fit`=元の視点のまま人物が画面いっぱいに映るよう自動フレーミング / `original`=元動画と同じ画角 |
| `VIEW_AZIMUTH_DEG` | マネキンを縦軸まわりに回して別角度から見る（度） |
| `SIDE_BY_SIDE` | 出力の左に元動画、右にマネキンを並べる（幅が 2 倍になります）。位置をそのまま見比べたいときは `CAMERA_MODE='original'` と併用 |

In [ ]:
#@title 5. 区間・出力の設定 { display-mode: "form" }
# --- 切り出す区間と入力の解像度 ---
START_SEC = 0.0  #@param {type:"number"}
END_SEC = 8.0  #@param {type:"number"}
TARGET_FPS = 30  #@param {type:"integer"}
MAX_HEIGHT = 720  #@param {type:"integer"}
PERSON_SELECT = "largest"  #@param ["largest", "center"]

# --- 推論（GPU が余っているなら BATCH_SIZE → NUM_AUG の順に増やす） ---
BATCH_SIZE = 8  #@param {type:"integer"}
NUM_AUG = 3  #@param [1, 3, 5, 7] {type:"raw"}
ANTIALIAS = 2  #@param [1, 2, 4] {type:"raw"}
DETECTOR_THRESHOLD = 0.25  #@param {type:"number"}

# --- モーションの整形（震え・足滑り対策） ---
MEDIAN_WINDOW = 3  #@param {type:"integer"}
SMOOTH_CUTOFF_HZ = 6.0  #@param {type:"number"}
ROOT_CUTOFF_HZ = 3.0  #@param {type:"number"}
DEPTH_EXTRA_SMOOTH = False  #@param {type:"boolean"}
FOOT_LOCK = True  #@param {type:"boolean"}
ROOT_MOTION = "locked"  #@param ["locked", "smoothed", "full"]

# --- 出力 ---
MANNEQUIN_STYLE = "auto"  #@param ["auto", "parts", "smpl_mesh"]
CAMERA_MODE = "fit"  #@param ["fit", "original"]
VIEW_AZIMUTH_DEG = 0  #@param {type:"slider", min:-180, max:180, step:15}
SHOW_FLOOR = True  #@param {type:"boolean"}
SIDE_BY_SIDE = False  #@param {type:"boolean"}
OUT_HEIGHT = 720  #@param {type:"integer"}

# ---- ffmpeg まわりのユーティリティ（imageio-ffmpeg 同梱のバイナリを使う） ----
import imageio.v2 as imageio
import imageio_ffmpeg


def ffmpeg_exe():
    try:
        return imageio_ffmpeg.get_ffmpeg_exe()
    except Exception:
        return 'ffmpeg'


def run_ffmpeg(args, check=True):
    p = subprocess.run([ffmpeg_exe(), '-hide_banner', *args],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if check and p.returncode != 0:
        raise RuntimeError('ffmpeg 失敗:\n' + p.stdout[-3000:])
    return p


def probe(path):
    with imageio.get_reader(path) as r:
        meta = r.get_meta_data()
    info = run_ffmpeg(['-i', path], check=False).stdout
    return dict(fps=float(meta.get('fps') or 30.0),
                duration=float(meta.get('duration') or 0.0),
                size=tuple(meta.get('size') or (0, 0)),
                has_audio=('Audio:' in info))


SRC_INFO = probe(VIDEO_PATH)
print('入力動画:', SRC_INFO)

duration = SRC_INFO['duration']
START_SEC = max(0.0, float(START_SEC))
END_SEC = float(END_SEC)
if END_SEC <= 0 or (duration and END_SEC > duration):
    END_SEC = duration if duration else END_SEC
assert END_SEC > START_SEC, '終点秒数は始点秒数より後にしてください。'
FPS = float(TARGET_FPS) if TARGET_FPS and TARGET_FPS > 0 else SRC_INFO['fps']
n_expected = int(round((END_SEC - START_SEC) * FPS))
print(f'切り出す区間: {START_SEC:.2f} 秒 〜 {END_SEC:.2f} 秒'
      f'（{END_SEC - START_SEC:.2f} 秒 / 約 {n_expected} フレーム @ {FPS:g} fps）')
if n_expected > 900:
    print('⚠️ フレーム数が多いので時間がかかります。まずは短い区間で試すことをおすすめします。')

---
## 6. 指定区間の切り出し（映像 + 音声）

映像は推論しやすいように `MAX_HEIGHT` 以下に縮小し、`TARGET_FPS` に揃えます。
音声は同じ区間を `m4a` として取り出し、最後にマネキン動画へ合成します（音声トラックが無い動画でもそのまま進みます）。

In [ ]:
#@title 6. 区間を切り出す { display-mode: "form" }
SEGMENT_MP4 = os.path.join(WORK_DIR, 'segment.mp4')
SEGMENT_AUDIO = os.path.join(WORK_DIR, 'segment.m4a')

vf = [f'fps={FPS}']
if MAX_HEIGHT and MAX_HEIGHT > 0:
    # 高さが MAX_HEIGHT を超えるときだけ縮小（幅は 2 の倍数に丸める）
    vf.append(f"scale='trunc(iw*min(1,{MAX_HEIGHT}/ih)/2)*2':'trunc(ih*min(1,{MAX_HEIGHT}/ih)/2)*2'")

run_ffmpeg(['-y', '-loglevel', 'error', '-ss', f'{START_SEC:.3f}', '-to', f'{END_SEC:.3f}',
            '-i', VIDEO_PATH, '-an', '-vf', ','.join(vf),
            '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '18', '-pix_fmt', 'yuv420p',
            SEGMENT_MP4])
SEG_INFO = probe(SEGMENT_MP4)
print('切り出した映像:', SEG_INFO)

AUDIO_PATH = None
if SRC_INFO['has_audio']:
    try:
        run_ffmpeg(['-y', '-loglevel', 'error', '-ss', f'{START_SEC:.3f}', '-to', f'{END_SEC:.3f}',
                    '-i', VIDEO_PATH, '-vn', '-c:a', 'aac', '-b:a', '192k', SEGMENT_AUDIO])
        if os.path.getsize(SEGMENT_AUDIO) > 0:
            AUDIO_PATH = SEGMENT_AUDIO
    except Exception as e:
        print('音声の取り出しに失敗しました:', repr(e))
print('音声:', AUDIO_PATH or 'なし（無音の動画を出力します）')

---
## 7. モーション抽出（NLF 推論）

各フレームを NLF に通して、人物ごとに

* `pose` … SMPL の関節回転（回転ベクトル 24×3）
* `betas` … 体型パラメータ（10 次元）
* `trans` … 全身の位置（m）
* `joints3d` / `vertices3d` … カメラ座標系の 3D 関節・頂点（mm 単位、x=右 / y=下 / z=奥）

を得ます。NLF は内部で **SMPLFitter** を使い、推定した非パラメトリックな頂点・関節に SMPL を当てはめています。

複数人が写っている場合は、**最初のフレームで選んだ人物に最も近い検出を毎フレーム追跡**して 1 人分だけを取り出します。
検出できなかったフレームは後で前後から補間します。

In [ ]:
#@title 7. NLF でモーションを抽出 { display-mode: "form" }
from tqdm.auto import tqdm

BODY_MODEL_NAME = 'smpl'  # smplx も指定できますが、このノートブックは smpl 前提です
N_VERTS, N_JOINTS = 6890, 24
N_PREVIEW = 4  # あとで重ね描画チェックに使うフレーム数

# クロップ単位のチャンクサイズ。num_aug より小さいとチャンク分割が無効になり
# 一気に全クロップを流してしまう（メモリ不足の原因）ので下限を設ける。
INTERNAL_BATCH_SIZE = max(64, int(NUM_AUG) * 4)
print(f'1 回の呼び出しで GPU に流すクロップ: 最大 {int(BATCH_SIZE) * int(NUM_AUG)} 枚 '
      f'(BATCH_SIZE={BATCH_SIZE} x NUM_AUG={NUM_AUG}, チャンク上限 {INTERNAL_BATCH_SIZE})')


def pick_first_person(boxes, image_w):
    # boxes: (n, 5) = x, y, w, h, score
    areas = boxes[:, 2] * boxes[:, 3]
    if PERSON_SELECT == 'center':
        # ある程度大きく写っている人の中で、画面中央に最も近い人を選ぶ
        big = np.flatnonzero(areas >= 0.3 * areas.max())
        cx = boxes[big, 0] + boxes[big, 2] / 2
        return int(big[np.argmin(np.abs(cx - image_w / 2))])
    return int(np.argmax(areas))


frames_pose, frames_betas, frames_trans = [], [], []
frames_joints, frames_verts, frames_box, frames_uncert = [], [], [], []
valid = []
preview = {}

reader = imageio.get_reader(SEGMENT_MP4)
prev_trans = None
batch_imgs, batch_idx = [], []
frame_count = 0
preview_at = set(np.linspace(0, max(n_expected - 1, 0), N_PREVIEW).astype(int).tolist())


def flush(batch_imgs, batch_idx):
    global prev_trans
    if not batch_imgs:
        return
    try:
        images = torch.from_numpy(np.stack(batch_imgs)).permute(0, 3, 1, 2).contiguous().to(DEVICE)
        with torch.inference_mode(), torch.device(DEVICE):
            pred = nlf_model.detect_smpl_batched(
                images, model_name=BODY_MODEL_NAME,
                detector_threshold=float(DETECTOR_THRESHOLD),
                internal_batch_size=INTERNAL_BATCH_SIZE, num_aug=int(NUM_AUG),
                antialias_factor=int(ANTIALIAS))
    except RuntimeError as e:
        # メモリ不足のときはバッチを半分に割ってやり直す
        if 'out of memory' not in str(e).lower() or len(batch_imgs) == 1:
            raise
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
        half = len(batch_imgs) // 2
        flush(batch_imgs[:half], batch_idx[:half])
        flush(batch_imgs[half:], batch_idx[half:])
        return
    for k in range(len(batch_idx)):
        boxes = pred['boxes'][k].detach().float().cpu().numpy()
        n_people = len(boxes)
        if n_people == 0:
            frames_pose.append(np.zeros(72, np.float32))
            frames_betas.append(np.zeros(10, np.float32))
            frames_trans.append(np.zeros(3, np.float32))
            frames_joints.append(np.zeros((N_JOINTS, 3), np.float32))
            frames_verts.append(np.zeros((N_VERTS, 3), np.float32))
            frames_box.append(np.zeros(5, np.float32))
            frames_uncert.append(np.nan)
            valid.append(False)
            continue
        trans = pred['trans'][k].detach().float().cpu().numpy()
        if prev_trans is None:
            j = pick_first_person(boxes, images.shape[3])
        else:
            d = np.linalg.norm(trans - prev_trans[None], axis=-1)
            j = int(np.argmin(d))
            if d[j] > 1.5:  # 追跡が外れたと判断したら一番大きい人に戻す
                j = int(np.argmax(boxes[:, 2] * boxes[:, 3]))
        prev_trans = trans[j]
        frames_pose.append(pred['pose'][k][j].detach().float().reshape(-1).cpu().numpy())
        frames_betas.append(pred['betas'][k][j].detach().float().cpu().numpy())
        frames_trans.append(trans[j])
        frames_joints.append(pred['joints3d'][k][j].detach().float().cpu().numpy())
        frames_verts.append(pred['vertices3d'][k][j].detach().float().cpu().numpy())
        frames_box.append(boxes[j])
        frames_uncert.append(
            float(pred['joint_uncertainties'][k][j].detach().float().mean().cpu()))
        valid.append(True)


pbar = tqdm(total=n_expected, desc='推論中')
for frame in reader:
    frame = np.asarray(frame)[..., :3]
    if frame_count in preview_at:
        preview[frame_count] = frame.copy()
    batch_imgs.append(frame)
    batch_idx.append(frame_count)
    frame_count += 1
    if len(batch_imgs) >= max(1, int(BATCH_SIZE)):
        flush(batch_imgs, batch_idx)
        pbar.update(len(batch_idx))
        batch_imgs, batch_idx = [], []
flush(batch_imgs, batch_idx)
pbar.update(len(batch_idx))
pbar.close()
reader.close()

valid = np.array(valid, bool)
motion_raw = dict(
    pose=np.stack(frames_pose).reshape(len(valid), -1, 3),
    betas=np.stack(frames_betas),
    trans=np.stack(frames_trans),
    joints3d=np.stack(frames_joints),
    vertices3d=np.stack(frames_verts),
    boxes=np.stack(frames_box),
)
N_FRAMES = len(valid)
UNCERTAINTY = np.array(frames_uncert, np.float32)
print(f'{N_FRAMES} フレーム処理 / 人物を検出できたフレーム: {int(valid.sum())}')
assert valid.any(), '人物が 1 人も検出できませんでした。区間や検出しきい値を変えてみてください。'
print('pose:', motion_raw['pose'].shape, ' vertices3d:', motion_raw['vertices3d'].shape, '(mm)')
if valid.any():
    u = UNCERTAINTY[valid]
    print(f'関節の推定不確実性: 中央値 {np.median(u):.0f} mm / 最大 {u.max():.0f} mm '
          f'(大きいほど推定が不安定なフレーム)')

---
## 8. モーションデータの整形と保存

NLF は 1 フレームずつ独立に推定するため、そのままだと**体が細かく震えます**。
ここでは頂点ではなく **SMPL のパラメータ（関節の回転・全身の位置・体型）を整えてから、
体モデルで頂点を作り直します**。こうすると骨の長さが厳密に一定になり、手足の伸び縮みによる震えが消えます。

1. 人物が検出できなかったフレームを前後から線形補間
2. **体型 `betas` をシーケンス全体の中央値 1 本に固定**（フレームごとの体型のゆらぎを除去）
3. **中央値フィルタ**（`MEDIAN_WINDOW`）で単発の外れフレームを除去
4. **ゼロ位相ローパス**（`scipy.signal.filtfilt`）で平滑化。前後両方向にかけるので**動きが遅れません**
   * 手足の回転と全身の位置 … `SMOOTH_CUTOFF_HZ`（既定 6 Hz）
   * **全身の向き** … `ROOT_CUTOFF_HZ`（既定 3 Hz）。胴の細かい震えはここが一番効きます
   * 奥行き `z` … `DEPTH_EXTRA_SMOOTH` が有効なら少しだけ強めに（3 Hz）
   * 位置を強く平滑化しすぎると、踏み替え（毎秒 2〜3 回）の動きまで削れて**かえって足が滑る**ため、
     位置のドリフトはフィルタではなく 8c の接地補正で直します
   * 回転は回転行列に直してから平滑化し、SVD で直交行列に戻しています
5. 整えたパラメータから**頂点と関節を再計算**（再ポーズ）
   * SMPL 公式ファイルは**不要**です。NLF の TorchScript の中に SMPL の本体（`body_models`）が
     入っているので、それをそのまま呼び出します
   * 念のため、**平滑化前**のパラメータで再ポーズしてセル 7 の結果と一致するか自己検証します。
     一致しなければ自動的に「頂点を直接平滑化する」従来方式にフォールバックします
6. `motion.npz` に保存

`motion.npz` が「抽出できたモーションデータ」です。他のツールで使いたいときはこのファイルを読み込んでください。

In [ ]:
#@title 8. 外れ値除去・平滑化・再ポーズして motion.npz に保存 { display-mode: "form" }
from scipy.ndimage import gaussian_filter1d, median_filter
from scipy.signal import butter, sosfiltfilt
from scipy.spatial.transform import Rotation

SMPL_PARENTS = np.array(
    [-1, 0, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 9, 9, 12, 13, 14, 16, 17, 18, 19, 20, 21], np.int32)
SMPL_JOINT_NAMES = [
    'pelvis', 'left_hip', 'right_hip', 'spine1', 'left_knee', 'right_knee', 'spine2',
    'left_ankle', 'right_ankle', 'spine3', 'left_foot', 'right_foot', 'neck', 'left_collar',
    'right_collar', 'head', 'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow',
    'left_wrist', 'right_wrist', 'left_hand', 'right_hand']
FOOT_JOINTS = {'left': [7, 10], 'right': [8, 11]}   # 足首とつま先


def fill_gaps(arr, valid):
    # 検出できなかったフレームを、前後の有効フレームから線形補間する
    arr = np.asarray(arr, np.float32)
    idx = np.arange(len(arr))
    vi = idx[valid]
    flat = arr.reshape(len(arr), -1)[valid]
    w = np.interp(idx, vi, np.arange(len(vi)))
    lo = np.floor(w).astype(int)
    hi = np.ceil(w).astype(int)
    t = (w - lo)[:, None].astype(np.float32)
    out = flat[lo] * (1 - t) + flat[hi] * t
    return out.reshape(arr.shape)


def lowpass(x, cutoff_hz, fps, order=2):
    # ゼロ位相 Butterworth ローパス（filtfilt は前後両方向にかけるので位相遅れが出ない）
    x = np.asarray(x, np.float32)
    if not cutoff_hz or cutoff_hz <= 0 or cutoff_hz >= fps / 2:
        return x
    sos = butter(order, float(cutoff_hz) / (fps / 2), 'low', output='sos')
    if len(x) <= 3 * (2 * len(sos) + 1):
        # フレーム数が少なすぎて filtfilt が使えないので、等価なガウシアンで代用
        return gaussian_filter1d(x, fps / (2 * np.pi * cutoff_hz), axis=0,
                                 mode='nearest').astype(np.float32)
    return np.ascontiguousarray(sosfiltfilt(sos, x, axis=0), dtype=np.float32)


def median_time(x, window):
    # 時間軸だけに中央値フィルタをかけて、単発の外れフレームを落とす
    x = np.asarray(x, np.float32)
    w = int(window) | 1   # 偶数なら奇数に
    if w < 3:
        return x
    size = [1] * x.ndim
    size[0] = w
    return median_filter(x, size=size, mode='nearest')


def orthonormalize(mats):
    shape = mats.shape
    u, _, vt = np.linalg.svd(mats.reshape(-1, 3, 3))
    det = np.linalg.det(u @ vt)
    u[det < 0, :, -1] *= -1
    return (u @ vt).reshape(shape).astype(np.float32)


def clean_rotations(rotvecs, valid, fps, window, cutoff_body, cutoff_root):
    # 回転ベクトル (T, J, 3) は回転行列に直してから扱う。
    # 重要: 回転ベクトルのまま補間・平滑化してはいけない。カメラ座標系では全身の向きが
    # ほぼ 180 度回転（|回転ベクトル| ≒ π）で、表現の切れ目にちょうど乗っているため、
    # 符号が反転したフレームをまたいで線形補間すると体が 1 回転してしまう。
    rv = np.asarray(rotvecs, np.float32)
    T, J = rv.shape[:2]
    mats = Rotation.from_rotvec(rv.reshape(-1, 3)).as_matrix().reshape(T, J, 3, 3)
    mats = orthonormalize(fill_gaps(mats, valid))   # 補間も回転行列の空間で行う
    mats = orthonormalize(median_time(mats, window))
    mats = np.concatenate(
        [lowpass(mats[:, :1], cutoff_root, fps),    # 関節 0 = 全身の向き
         lowpass(mats[:, 1:], cutoff_body, fps)], axis=1)
    mats = orthonormalize(mats)
    return Rotation.from_matrix(mats.reshape(-1, 3, 3)).as_rotvec().reshape(T, J, 3).astype(
        np.float32)


def jitter_metric(joints):
    # 2 階差分の大きさ = 細かい震えの指標 [mm]
    if len(joints) < 3:
        return float('nan')
    d2 = joints[2:] - 2 * joints[1:-1] + joints[:-2]
    return float(np.linalg.norm(d2, axis=-1).mean())


def get_repose_fn():
    # 整えたパラメータから頂点・関節を作り直す関数を用意する
    try:
        bm = getattr(nlf_model.body_models, BODY_MODEL_NAME)

        def fn(pose, betas, trans):
            with torch.inference_mode():
                out = bm(pose_rotvecs=pose, shape_betas=betas, trans=trans)
            return out['vertices'], out['joints']

        return fn, 'NLF の TorchScript 内の SMPL（公式ファイル不要）'
    except Exception as e:
        print('TorchScript 内の体モデルを取り出せませんでした:', repr(e))

    if BODY_MODEL is not None:
        bm2 = BODY_MODEL.to(DEVICE)

        def fn(pose, betas, trans):
            with torch.inference_mode():
                out = bm2(pose_rotvecs=pose, shape_betas=betas, trans=trans)
            return out['vertices'], out['joints']

        return fn, 'smplfitter の公式 SMPL'
    return None, None


def repose(pose, betas, trans, fn, chunk=256):
    # 戻り値は mm（体モデルは m を返すので 1000 倍する）
    verts, joints = [], []
    for st in range(0, len(pose), chunk):
        sl = slice(st, st + chunk)
        p = torch.from_numpy(np.ascontiguousarray(pose[sl])).float().to(DEVICE)
        b = torch.from_numpy(np.ascontiguousarray(betas[sl])).float().to(DEVICE)
        t = torch.from_numpy(np.ascontiguousarray(trans[sl])).float().to(DEVICE)
        v, j = fn(p.reshape(len(p), -1), b, t)
        verts.append(v.float().cpu().numpy() * 1000.0)
        joints.append(j.float().cpu().numpy() * 1000.0)
    return np.concatenate(verts), np.concatenate(joints)


# ---- 1. 欠損補間（pose だけは clean_rotations の中で回転行列として補間する） ----
motion = {k: (v.copy() if k == 'pose' else fill_gaps(v, valid))
          for k, v in motion_raw.items()}

# ---- 2. 体型を 1 本に固定 ----
betas_const = np.median(motion_raw['betas'][valid], axis=0).astype(np.float32)
motion['betas'] = np.tile(betas_const, (N_FRAMES, 1))

# ---- 3-4. 中央値フィルタ＋ゼロ位相ローパス ----
motion['pose'] = clean_rotations(
    motion['pose'], valid, FPS, MEDIAN_WINDOW, SMOOTH_CUTOFF_HZ, ROOT_CUTOFF_HZ)
# 全身の「位置」は関節と同じ遮断周波数で。ここを下げすぎると歩幅に相当する
# 本物の動き（毎秒 2〜3 回の踏み替え）まで削れてしまい、かえって足が滑る。
# 位置のドリフトは 8c の接地補正で直す。
trans = lowpass(median_time(motion['trans'], MEDIAN_WINDOW), SMOOTH_CUTOFF_HZ, FPS)
if DEPTH_EXTRA_SMOOTH:
    # 奥行きだけは単眼推定で特に不安定なので、少しだけ強めに平滑化する
    trans[:, 2] = lowpass(trans[:, 2:3], min(3.0, SMOOTH_CUTOFF_HZ), FPS)[:, 0]
motion['trans'] = trans

# ---- 5. 再ポーズ（できなければ頂点を直接平滑化） ----
repose_fn, repose_name = get_repose_fn()
if repose_fn is not None:
    check = np.flatnonzero(valid)[:8]
    try:
        v_chk, _ = repose(motion_raw['pose'][check], motion_raw['betas'][check],
                          motion_raw['trans'][check], repose_fn, chunk=8)
        err = float(np.abs(v_chk - motion_raw['vertices3d'][check]).max())
        print(f'再ポーズの自己検証: セル 7 の頂点との最大差 {err:.3f} mm（{repose_name}）')
        if not np.isfinite(err) or err > 1.0:
            print('⚠️ 一致しなかったため、再ポーズは使わず頂点を直接平滑化します。')
            repose_fn = None
    except Exception as e:
        print('⚠️ 再ポーズを実行できませんでした:', repr(e))
        repose_fn = None

if repose_fn is not None:
    motion['vertices3d'], motion['joints3d'] = repose(
        motion['pose'], motion['betas'], motion['trans'], repose_fn)
else:
    # フォールバック: 頂点・関節に直接フィルタをかける。
    # 全身の位置だけは骨盤の軌跡から差分を作って、より低いカットオフを適用する。
    verts = lowpass(median_time(motion['vertices3d'], MEDIAN_WINDOW), SMOOTH_CUTOFF_HZ, FPS)
    joints = lowpass(median_time(motion['joints3d'], MEDIAN_WINDOW), SMOOTH_CUTOFF_HZ, FPS)
    root = joints[:, 0]
    root_target = root.copy()
    if DEPTH_EXTRA_SMOOTH:
        root_target[:, 2] = lowpass(root[:, 2:3], min(3.0, SMOOTH_CUTOFF_HZ), FPS)[:, 0]
    delta = (root_target - root)[:, None]
    motion['vertices3d'] = verts + delta
    motion['joints3d'] = joints + delta

print(f'震えの指標（関節の 2 階差分）: 整形前 {jitter_metric(motion_raw["joints3d"]):.2f} mm '
      f'-> 整形後 {jitter_metric(motion["joints3d"]):.2f} mm')

# ---- 6. 保存 ----
MOTION_NPZ = os.path.join(WORK_DIR, 'motion.npz')


def save_motion():
    # motion を書き換えるセル（8b, 8c）の最後でも呼ぶこと。
    # そうしないと .npz と描画結果が食い違う。
    np.savez_compressed(
        MOTION_NPZ,
        pose=motion['pose'].astype(np.float32),              # (T, 24, 3) 回転ベクトル [rad]
        betas=motion['betas'].astype(np.float32),            # (T, 10)
        betas_const=betas_const,                             # (10,) シーケンス共通の体型
        trans=motion['trans'].astype(np.float32),            # (T, 3) [m]
        joints3d=motion['joints3d'].astype(np.float32),      # (T, 24, 3) [mm] カメラ座標系
        vertices3d=motion['vertices3d'].astype(np.float32),  # (T, 6890, 3) [mm]
        ground_offset_mm=motion.get(
            'ground_offset_mm', np.zeros((N_FRAMES, 3), np.float32)),
        valid=valid,
        joint_uncertainty=UNCERTAINTY,                       # (T,) [mm]
        repose_ok=np.bool_(repose_fn is not None),
        fps=np.float32(FPS),
        start_sec=np.float32(START_SEC),
        end_sec=np.float32(END_SEC),
        smooth_cutoff_hz=np.float32(SMOOTH_CUTOFF_HZ),
        root_cutoff_hz=np.float32(ROOT_CUTOFF_HZ),
        median_window=np.int32(MEDIAN_WINDOW),
        num_aug=np.int32(NUM_AUG),
        joint_names=np.array(SMPL_JOINT_NAMES),
        kintree_parents=SMPL_PARENTS,
        body_model=np.array(BODY_MODEL_NAME),
    )
    return MOTION_NPZ


save_motion()
print('モーションデータを保存しました:', MOTION_NPZ,
      f'({os.path.getsize(MOTION_NPZ) / 1024 ** 2:.1f} MB)')
print('共通の体型 betas:', np.round(betas_const, 3))

### 8b.（任意）SMPLFitter で全フレーム共通の体型に整える

SMPL 公式ファイルがある場合のみの**代替手段**です（既定はオフ）。
セル 8 で体型はすでに中央値 1 本に固定しているので通常は不要ですが、
[SMPLFitter](https://github.com/isarandi/smplfitter) の `share_beta=True` を使うと、
体型を共有したまま**頂点から SMPL パラメータを当てはめ直す**ことができます。
公式ファイルを持っていて、体型の推定をやり直したい場合に有効化してください。

In [ ]:
#@title 8b. 体型を共通化するリフィット（公式 SMPL がある場合のみ） { display-mode: "form" }
REFIT_SHARED_SHAPE = False  #@param {type:"boolean"}

if REFIT_SHARED_SHAPE and BODY_MODEL is not None:
    from smplfitter.pt import BodyFitter
    bm = BODY_MODEL.to(DEVICE)
    fitter = BodyFitter(bm).to(DEVICE)
    verts_t = torch.from_numpy(motion['vertices3d'] / 1000.0).float().to(DEVICE)
    joints_t = torch.from_numpy(motion['joints3d'] / 1000.0).float().to(DEVICE)
    fits = []
    # 体型はバッチ内で共有されるため、可能な限り 1 回で全フレームを流す
    # （チャンクに分けると境界ごとに体型が変わり、周期的な段差が出る）
    chunk = len(verts_t) if len(verts_t) <= 512 else 64
    with torch.inference_mode():
        for s in range(0, len(verts_t), chunk):
            fits.append(fitter.fit(target_vertices=verts_t[s:s + chunk],
                                   target_joints=joints_t[s:s + chunk],
                                   num_iter=3, beta_regularizer=1.0, share_beta=True,
                                   final_adjust_rots=True,
                                   requested_keys=['pose_rotvecs', 'shape_betas', 'trans']))
        pose_rotvecs = torch.cat([f['pose_rotvecs'] for f in fits])
        shape_betas = torch.cat([f['shape_betas'] for f in fits])
        trans = torch.cat([f['trans'] for f in fits])
        shape_betas = shape_betas.mean(0, keepdim=True).expand_as(shape_betas).contiguous()
        out = bm(pose_rotvecs=pose_rotvecs, shape_betas=shape_betas, trans=trans)
    motion['pose'] = pose_rotvecs.reshape(len(verts_t), -1, 3).cpu().numpy()
    motion['betas'] = shape_betas.cpu().numpy()
    motion['trans'] = trans.cpu().numpy()
    motion['vertices3d'] = lowpass(out['vertices'].cpu().numpy() * 1000.0, SMOOTH_CUTOFF_HZ, FPS)
    motion['joints3d'] = lowpass(out['joints'].cpu().numpy() * 1000.0, SMOOTH_CUTOFF_HZ, FPS)
    motion.pop('ground_offset_mm', None)   # 接地補正はやり直しになる
    save_motion()
    print('共通体型でリフィットしました。betas =', np.round(motion['betas'][0], 3))
else:
    print('スキップしました（公式 SMPL ファイルが無い、または REFIT_SHARED_SHAPE=False）。')

### 8c. 接地補正（足の滑りを取る）

単眼推定では**奥行きが最も不安定**で、さらに元動画のカメラの動きもそのまま全身の平行移動として出てしまいます。
その結果、足が地面をツルツル滑って見えます。ここでは

1. 足関節（足首・つま先）の高さから**床の高さ**を推定し、
2. 「**床に近く・水平方向にほとんど動いていない**」フレームを**接地**とみなし、
3. 接地している足が地面上で止まるように、**全身の位置**（平行移動）を補正します。

補正は平行移動だけなので、関節角度（ポーズ）には手を加えません。足の踏み替えは保たれます。

`ROOT_MOTION` で全身の移動の扱いを選べます。

| 値 | 動作 |
|---|---|
| `locked` | 低周波のドリフトだけを除去して**その場で踊る**。カメラのパンや奥行きドリフトが原因の滑りに最も効きます（既定） |
| `smoothed` | 移動は残しつつ水平方向の軌跡を 1 Hz で平滑化。接地補正の細かい成分も一部削れます |
| `full` | 接地補正後の移動をそのまま使います |

グラフの横軸は時間（フレーム）です。上段が足の床からの高さ（灰色の帯が接地と判定した区間）、
下段が足の水平速度で、**接地区間の速度が下がっていれば補正が効いています**。

In [ ]:
#@title 8c. 接地補正と全身の移動の扱い { display-mode: "form" }
import matplotlib.pyplot as plt

CONTACT_HEIGHT_MM = 80.0  #@param {type:"number"}
CONTACT_SPEED_MMPS = 350.0  #@param {type:"number"}
LOCK_LEAK_SEC = 5.0  #@param {type:"number"}
LOCK_CUTOFF_HZ = 0.3  #@param {type:"number"}

CONTACT_JOINTS = [7, 8, 10, 11]   # 左足首, 右足首, 左つま先, 右つま先
CONTACT_NAMES = ['L ankle', 'R ankle', 'L toe', 'R toe']


def smoothstep(x):
    x = np.clip(x, 0.0, 1.0)
    return x * x * (3.0 - 2.0 * x)


def horizontal_speed(p, fps):
    # 水平（x, z）方向の速さ [mm/s]。中央差分。
    v = np.empty(p.shape[:-1], np.float32)
    d = (p[2:] - p[:-2])[..., [0, 2]]
    v[1:-1] = np.linalg.norm(d, axis=-1) * (fps / 2.0)
    v[0], v[-1] = v[1], v[-2]
    return v


# 前回の補正を取り消してから計算する（セルを何度実行しても結果が同じになるように）
prev_off = motion.pop('ground_offset_mm', None)
if prev_off is not None:
    motion['vertices3d'] = motion['vertices3d'] - prev_off[:, None]
    motion['joints3d'] = motion['joints3d'] - prev_off[:, None]
    motion['trans'] = motion['trans'] - prev_off / 1000.0

if N_FRAMES < 12:
    print('フレーム数が少なすぎるため接地補正はスキップします。')
else:
    jnt = motion['joints3d']
    P = jnt[:, CONTACT_JOINTS]                       # (T, 4, 3) [mm]

    def slow(x, fc):
        # 遮断周波数が低いので、端で振れる Butterworth ではなくガウシアンを使う
        return gaussian_filter1d(np.asarray(x, np.float32),
                                 FPS / (2 * np.pi * max(fc, 1e-3)), axis=0, mode='nearest')

    # 接地点ごとの「地面に着いているときの高さ」を基準にする。
    # 足首はつま先より数 cm 高い位置にあるので、床を 1 枚の高さで代表させると
    # 足首がいつまでも「浮いている」と判定されてしまう。
    ref_y = np.percentile(P[..., 1], 75, axis=0)     # (4,) y は下向き = 大きいほど低い
    height = (ref_y[None] - P[..., 1]).astype(np.float32)   # 各点の基準面からの高さ [mm]
    eligible = (ref_y.max() - ref_y) < 150.0         # 一度も床付近に来ない点は除外
    speed = horizontal_speed(P, FPS)                 # (T, 4) [mm/s]

    # 速さは「絶対値」ではなく「そのフレームで一番遅い足との差」で見る。
    # カメラが動いている動画ではカメラ座標系で完全に止まる足は存在しないので、
    # 絶対値でしきい値を切ると接地が 1 つも検出できなくなる。
    v_floor = np.min(np.where(eligible[None], speed, np.inf), axis=1, keepdims=True)
    v_rel = speed - v_floor

    # 接地の重み（0〜1）。硬い 0/1 判定にすると接地の切り替わりで補正が飛ぶ。
    w = (smoothstep(1.0 - height / CONTACT_HEIGHT_MM)
         * smoothstep(1.0 - v_rel / CONTACT_SPEED_MMPS)
         # 一番遅い足まで速いフレーム（ジャンプ中など）は全部「非接地」にする
         * smoothstep(1.0 - v_floor / (4.0 * CONTACT_SPEED_MMPS)))
    # 一度も床付近に来ない点と、人物を検出できなかった（補間した）フレームは信用しない。
    # 重みが連続値なので、単発の誤検出はここで 0/1 に丸めず、そのまま小さい重みとして扱う
    # （前後 2 フレーム連続で接地していないと補正に効かないため、これで十分）。
    w = (w * eligible[None] * valid[:, None]).astype(np.float32)
    # --- 水平方向: 接地している足が動かないように全身をずらす ---
    corr = np.zeros((N_FRAMES, 2), np.float32)
    if FOOT_LOCK:
        # 前後フレームとも接地している点だけを使う。重みは 2 乗して、
        # 「確実に着いている点」を優先する（浮きかけのつま先に引っ張られないように）。
        W = (w[1:] * w[:-1]) ** 2
        den = W.sum(-1)
        step = (W[..., None] * (P[1:] - P[:-1])[..., [0, 2]]).sum(1) / np.maximum(den, 1e-6)[:, None]
        delta = -step * smoothstep(den / 0.5)[:, None]
        delta[den <= 1e-6] = 0.0
        # 漏れ項つきの積分。時定数 LOCK_LEAK_SEC 秒で元の軌跡に戻るので補正が暴走しない。
        lam = float(np.exp(-1.0 / max(LOCK_LEAK_SEC * FPS, 1e-6)))
        acc = np.zeros(2, np.float32)
        for t in range(1, N_FRAMES):
            acc = lam * acc + delta[t - 1]
            n = float(np.linalg.norm(acc))
            if n > 1000.0:
                acc = acc * (1000.0 / n)
            corr[t] = acc
        # 強くローパスすると、せっかく止めた接地区間の中で滑りが戻ってしまうので、
        # 高めの遮断周波数で軽くだけ均す（重み w が連続なので段差は出ない）
        corr = lowpass(corr, min(8.0, FPS / 2 * 0.9), FPS)

    # --- 垂直方向: 接地している足を基準面に戻す（ジャンプは潰さない） ---
    dy = np.zeros(N_FRAMES, np.float32)
    if FOOT_LOCK:
        wsum = w.sum(-1)
        # e = 接地している点が基準面からどれだけ浮いているか（上向きが正）
        e = (w * height).sum(-1) / np.maximum(wsum, 1e-6)
        gate = np.clip(gaussian_filter1d(np.clip(wsum, 0, 1), 2.0, mode='nearest'), 0, 1)
        # y は下向きが正なので、e だけ浮いていれば +e ずらせば基準面に戻る。
        # 滞空中は gate=0 なので補正は 0 になり、ジャンプの上下動はそのまま残る。
        dy = lowpass((e * gate)[:, None], min(3.0, ROOT_CUTOFF_HZ), FPS)[:, 0]
        # フィルタで滞空中ににじんだ分を gate で戻す。省くと踏み切りと着地が鈍る。
        dy = np.clip(dy * gate, -150.0, 150.0)

    # --- 全身の移動の扱い ---
    root_xz = jnt[:, 0][:, [0, 2]] + corr
    if ROOT_MOTION == 'locked':
        # 低周波のドリフト（カメラのパンや奥行きのずれ）だけを除去。
        # 全部消すと体重移動まで消えて、かえって足が滑って見える。
        off_xz = corr - (slow(root_xz, LOCK_CUTOFF_HZ) - root_xz.mean(0, keepdims=True))
    elif ROOT_MOTION == 'smoothed':
        off_xz = corr + (slow(root_xz, 1.0) - root_xz)
    else:
        off_xz = corr

    offset = np.stack([off_xz[:, 0], dy, off_xz[:, 1]], axis=1).astype(np.float32)
    motion['vertices3d'] = motion['vertices3d'] + offset[:, None]
    motion['joints3d'] = motion['joints3d'] + offset[:, None]
    motion['trans'] = motion['trans'] + offset / 1000.0
    motion['ground_offset_mm'] = offset

    # 床の高さ（足の裏 = 頂点ベース）を描画セルへ渡す
    contact_frames = np.flatnonzero(w.max(-1) > 0.5)
    if len(contact_frames) > 0:
        FLOOR_Y_MM = float(np.percentile(motion['vertices3d'][contact_frames][..., 1], 99.5))
    else:
        FLOOR_Y_MM = float(np.percentile(motion['vertices3d'][..., 1], 99.7))

    # --- 効果の確認 ---
    speed_after = horizontal_speed(motion['joints3d'][:, CONTACT_JOINTS], FPS)
    cm = w > 0.5
    print('接地率: ' + ' / '.join(
        f'{CONTACT_NAMES[i]} {100 * cm[:, i].mean():.0f}%' for i in range(4)))
    if cm.any():
        print(f'接地中の足の水平速度（中央値）: {np.median(speed[cm]):.0f} mm/s '
              f'-> {np.median(speed_after[cm]):.0f} mm/s')
    else:
        print('⚠️ 接地フレームが見つかりませんでした。CONTACT_HEIGHT_MM / '
              'CONTACT_SPEED_MMPS を大きくしてみてください。')
    print(f'補正量: 水平 最大 {np.abs(off_xz).max():.0f} mm / 垂直 平均 {np.abs(dy).mean():.0f} mm'
          f'（FOOT_LOCK={FOOT_LOCK}, ROOT_MOTION={ROOT_MOTION}）')
    save_motion()

    fig, axes = plt.subplots(2, 1, figsize=(9, 4.2), sharex=True)
    for i in range(4):
        axes[0].plot(height[:, i], lw=1, label=CONTACT_NAMES[i])
        axes[1].plot(speed[:, i], lw=0.8, alpha=0.35)
        axes[1].plot(speed_after[:, i], lw=1, label=CONTACT_NAMES[i])
    axes[0].fill_between(np.arange(N_FRAMES), 0, 1, where=cm.any(1),
                         transform=axes[0].get_xaxis_transform(), color='0.85', zorder=0)
    axes[0].axhline(CONTACT_HEIGHT_MM, color='r', lw=0.8, ls='--')
    axes[0].set_ylabel('height above plant [mm]')
    axes[0].set_title('shaded = detected contact | thin = before, thick = after', fontsize=9)
    axes[1].axhline(CONTACT_SPEED_MMPS, color='r', lw=0.8, ls='--')
    axes[1].set_ylabel('foot speed [mm/s]')
    axes[1].set_xlabel('frame')
    axes[1].set_ylim(0, max(CONTACT_SPEED_MMPS * 3, float(np.percentile(speed, 99))))
    axes[0].legend(fontsize=7, ncol=4)
    plt.tight_layout()
    plt.show()

---
## 9. 抽出結果の確認（元フレームへの重ね描画）

推定した 3D 頂点を元のフレームに投影して重ねます。人物にきれいに重なっていれば抽出は成功です。
ずれている場合は、区間を変える・`PERSON_SELECT` を変える・`MAX_HEIGHT` を上げる、などを試してください。

※ 8c の接地補正は体の位置を意図的にずらすため、この重ね描画では**補正前の位置**に戻して表示しています。

In [ ]:
#@title 9. 重ね描画でチェック { display-mode: "form" }
import matplotlib.pyplot as plt


def intrinsics_from_fov(fov_degrees, imshape):
    # NLF が既定で仮定しているカメラ（対角ではなく長辺基準の画角 55 度）と同じ式
    h, w = imshape[:2]
    f = float(max(h, w)) / (2.0 * np.tan(np.deg2rad(fov_degrees) / 2.0))
    return np.array([[f, 0, w / 2.0], [0, f, h / 2.0], [0, 0, 1]], np.float32)


def project(points_cam, K):
    z = np.maximum(points_cam[..., 2:], 1e-3)
    uv = points_cam[..., :2] / z
    return uv * np.array([K[0, 0], K[1, 1]], np.float32) + np.array([K[0, 2], K[1, 2]], np.float32)


SEG_W, SEG_H = SEG_INFO['size']
K_ORIG = intrinsics_from_fov(55.0, (SEG_H, SEG_W))

# 接地補正は体を「元の見え方」から意図的にずらすので、重ね描画では取り消して表示する
cam_verts = motion['vertices3d'] - motion.get(
    'ground_offset_mm', np.zeros((N_FRAMES, 3), np.float32))[:, None]

keys = sorted(k for k in preview if k < N_FRAMES)
if keys:
    fig, axes = plt.subplots(1, len(keys), figsize=(4 * len(keys), 4 * SEG_H / max(SEG_W, 1)))
    axes = np.atleast_1d(axes)
    for ax, k in zip(axes, keys):
        uv = project(cam_verts[k], K_ORIG)
        ax.imshow(preview[k])
        ax.scatter(uv[::12, 0], uv[::12, 1], s=1.0, c='lime', alpha=0.45)
        ax.set_title(f'frame {k}' + ('' if valid[k] else ' (interpolated)'), fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('プレビュー用のフレームがありません。')

---
## 10. マネキンのメッシュを組み立てる

**方式 A: SMPL メッシュ**（公式ファイルがある場合）
: SMPL の面情報（13,776 三角形）をそのまま使い、推定された頂点をフレームごとに差し替えます。

**方式 B: パーツ凸包マネキン**（既定 / 公式ファイル不要）
: SMPL の 6,890 頂点を体のパーツ（関節）ごとに分け、**パーツごとの凸包**を取って
  デッサン人形のような多面体マネキンを作ります。
  頂点の所属パーツは、TorchScript モデルの中に入っているスキニングウェイト（LBS weights）から取得し、
  取れない場合は「最も近い関節」で代用します。
  凸包の面の構成（トポロジ）は 1 フレームだけで計算し、以降のフレームは同じ面構成のまま頂点を差し替えるので高速です。

In [ ]:
#@title 10. マネキンのメッシュを作る { display-mode: "form" }
from scipy.spatial import ConvexHull


def try_get_lbs_weights(model, model_name, n_verts):
    # NLF の TorchScript には SMPL のバッファが入っているので、そこからスキニングウェイトを拝借する
    try:
        bm = getattr(model.body_models, model_name)
        w = bm.weights.detach().float().cpu().numpy()
        if w.ndim == 2 and w.shape[0] == n_verts:
            return w
    except Exception:
        pass
    return None


def vertex_part_labels(verts_ref, joints_ref, lbs_weights=None):
    if lbs_weights is not None:
        return lbs_weights.argmax(-1).astype(np.int32)
    d = np.linalg.norm(verts_ref[:, None, :] - joints_ref[None, :, :], axis=-1)
    return d.argmin(1).astype(np.int32)


def build_part_hull_topology(verts_ref, labels, num_parts, min_points=8):
    # パーツごとの凸包 -> 「頂点インデックス列 + 固定の面」に変換する
    idx_chunks, face_chunks, face_part = [], [], []
    offset = 0
    for p in range(num_parts):
        member = np.flatnonzero(labels == p)
        if len(member) < min_points:
            continue
        pts = verts_ref[member]
        try:
            hull = ConvexHull(pts, qhull_options='QJ')
        except Exception:
            continue
        used = np.unique(hull.simplices)
        remap = np.full(len(member), -1, np.int64)
        remap[used] = np.arange(len(used))
        tris = remap[hull.simplices]
        v = pts[used]
        a, b, c = v[tris[:, 0]], v[tris[:, 1]], v[tris[:, 2]]
        n = np.cross(b - a, c - a)
        flip = np.einsum('ij,ij->i', n, (a + b + c) / 3.0 - v.mean(0)) < 0
        tris[flip] = tris[flip][:, ::-1]
        idx_chunks.append(member[used])
        face_chunks.append(tris + offset)
        face_part.append(np.full(len(tris), p, np.int32))
        offset += len(used)
    return (np.concatenate(idx_chunks), np.concatenate(face_chunks).astype(np.int32),
            np.concatenate(face_part))


VERTS = motion['vertices3d']                 # (T, V, 3) [mm]
JOINTS = motion['joints3d']                  # (T, J, 3) [mm]
ref = int(np.flatnonzero(valid)[len(np.flatnonzero(valid)) // 2])  # 代表フレーム

use_smpl_mesh = (MANNEQUIN_STYLE == 'smpl_mesh'
                 or (MANNEQUIN_STYLE == 'auto' and SMPL_FACES is not None))
if use_smpl_mesh and SMPL_FACES is None:
    print('⚠️ SMPL 公式ファイルが無いのでパーツ凸包マネキンに切り替えます。')
    use_smpl_mesh = False

if use_smpl_mesh:
    VERTEX_MAP = np.arange(VERTS.shape[1])
    FACES = SMPL_FACES
    print(f'SMPL メッシュを使用します: 頂点 {len(VERTEX_MAP)} / 面 {len(FACES)}')
else:
    lbs = try_get_lbs_weights(nlf_model, BODY_MODEL_NAME, VERTS.shape[1])
    print('スキニングウェイト:', 'TorchScript から取得' if lbs is not None else '最近傍関節で代用')
    LABELS = vertex_part_labels(VERTS[ref], JOINTS[ref], lbs)
    VERTEX_MAP, FACES, FACE_PART = build_part_hull_topology(
        VERTS[ref], LABELS, num_parts=JOINTS.shape[1])
    print(f'パーツ凸包マネキン: パーツ {len(np.unique(FACE_PART))} / '
          f'頂点 {len(VERTEX_MAP)} / 面 {len(FACES)}')

MESH_VERTS = VERTS[:, VERTEX_MAP] / 1000.0   # (T, M, 3) [m]
print('マネキンの頂点列:', MESH_VERTS.shape)

---
## 11. レンダリング

カメラ座標系（x=右 / y=下 / z=奥）のまま描画します。

* `CAMERA_MODE='fit'` … 元動画と同じ視点のまま、シーケンス全体が画面に収まるよう焦点距離と中心を自動調整
* `CAMERA_MODE='original'` … 元動画とまったく同じ画角（人物の位置もそのまま）
* `VIEW_AZIMUTH_DEG` … 縦軸まわりに回転させて別アングルから撮影
* `SHOW_FLOOR` … 足元の最下点に市松模様の床を敷きます（奥行きが分かりやすくなります）

レンダラは OpenGL 不要の**ソフトウェアレンダラ**（三角形を奥から順に塗る画家アルゴリズム）です。
Colab でも追加インストール無しで確実に動きます。`pyrender` が使える環境なら `RENDER_BACKEND='pyrender'`
にすると、より陰影のきれいな描画になります（失敗したら自動でソフトウェアに戻ります）。

In [ ]:
#@title 11. マネキン動画をレンダリング { display-mode: "form" }
RENDER_BACKEND = "software"  #@param ["software", "pyrender"]
BODY_COLOR = "#D8D2C6"  #@param {type:"string"}
BG_COLOR = "#1C2029"  #@param {type:"string"}

from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.collections import PolyCollection
from matplotlib.figure import Figure


def hex2rgb(s):
    s = s.lstrip('#')
    return np.array([int(s[i:i + 2], 16) / 255.0 for i in (0, 2, 4)], np.float32)


def rotate_about_y(points, center, degrees):
    th = np.deg2rad(degrees)
    c, s = np.cos(th), np.sin(th)
    R = np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]], np.float32)  # y 軸（上下）まわり
    return (points - center) @ R.T + center


def fit_camera(points, imshape, margin=0.14):
    h, w = imshape[:2]
    p = points.reshape(-1, 3)
    p = p[p[:, 2] > 1e-3]
    u, v = p[:, 0] / p[:, 2], p[:, 1] / p[:, 2]
    u0, u1 = np.percentile(u, 0.1), np.percentile(u, 99.9)
    v0, v1 = np.percentile(v, 0.1), np.percentile(v, 99.9)
    f = min(w / max(u1 - u0, 1e-6), h / max(v1 - v0, 1e-6)) * (1.0 - margin)
    return np.array([[f, 0, w / 2 - f * (u0 + u1) / 2],
                     [0, f, h / 2 - f * (v0 + v1) / 2], [0, 0, 1]], np.float32)


def make_floor_grid(center_xz, y_level, half_size, n_cells=14):
    xs = np.linspace(center_xz[0] - half_size, center_xz[0] + half_size, n_cells + 1)
    zs = np.linspace(center_xz[1] - half_size, center_xz[1] + half_size, n_cells + 1)
    verts, faces, shade = [], [], []
    for i in range(n_cells):
        for j in range(n_cells):
            o = len(verts)
            verts += [[xs[i], y_level, zs[j]], [xs[i + 1], y_level, zs[j]],
                      [xs[i + 1], y_level, zs[j + 1]], [xs[i], y_level, zs[j + 1]]]
            faces += [[o, o + 1, o + 2], [o, o + 2, o + 3]]
            c = 0.80 if (i + j) % 2 == 0 else 0.66
            shade += [c, c]
    return (np.array(verts, np.float32), np.array(faces, np.int32),
            np.array(shade, np.float32)[:, None] * np.ones(3, np.float32))


def shade_faces(verts, faces, base_color, light_dir=(0.35, -0.75, -0.55),
                ambient=0.42, diffuse=0.72):
    a, b, c = verts[faces[:, 0]], verts[faces[:, 1]], verts[faces[:, 2]]
    n = np.cross(b - a, c - a)
    n = n / np.maximum(np.linalg.norm(n, axis=-1, keepdims=True), 1e-9)
    l = np.asarray(light_dir, np.float32)
    l = l / np.linalg.norm(l)
    lam = np.abs(n @ l)
    return np.clip(np.asarray(base_color, np.float32)[None] * (ambient + diffuse * lam)[:, None],
                   0, 1)


def render_software(verts, faces, K, imshape, body_color, bg_color, extra=None):
    h, w = imshape[:2]
    V, F, C = [verts], [faces], [shade_faces(verts, faces, body_color)]
    n_body_faces = len(faces)
    if extra is not None:
        ev, ef, ec = extra
        V.append(ev)
        F.append(ef + len(verts))
        C.append(np.clip(shade_faces(ev, ef, (1.0, 1.0, 1.0)) * ec, 0, 1))
    V, F, C = np.concatenate(V), np.concatenate(F), np.concatenate(C)
    tri = V[F]
    depth = tri[..., 2].mean(1)
    n = np.cross(tri[:, 1] - tri[:, 0], tri[:, 2] - tri[:, 0])
    facing = np.einsum('ij,ij->i', n, tri.mean(1)) < 0   # カメラ（原点）を向いている面だけ描く
    facing[n_body_faces:] = True                        # 床は両面描画
    keep = (depth > 1e-3) & facing
    F, C, depth = F[keep], C[keep], depth[keep]
    order = np.argsort(-depth)                          # 奥から手前へ
    polys = project(V, K)[F[order]]

    fig = Figure(figsize=(w / 100.0, h / 100.0), dpi=100)
    canvas = FigureCanvasAgg(fig)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, w)
    ax.set_ylim(h, 0)
    ax.axis('off')
    fig.patch.set_facecolor(bg_color)
    ax.set_facecolor(bg_color)
    ax.add_collection(PolyCollection(polys, facecolors=C[order], edgecolors='none'))
    canvas.draw()
    return np.asarray(canvas.buffer_rgba())[..., :3].copy()


class PyrenderBackend:
    def __init__(self, imshape, bg_color):
        os.environ.setdefault('PYOPENGL_PLATFORM', 'egl')
        import pyrender
        import trimesh
        self.pyrender, self.trimesh = pyrender, trimesh
        self.renderer = pyrender.OffscreenRenderer(imshape[1], imshape[0])
        self.bg = bg_color

    def render(self, verts, faces, K, imshape, body_color, bg_color, extra=None):
        pyrender, trimesh = self.pyrender, self.trimesh
        scene = pyrender.Scene(bg_color=[*bg_color, 1.0], ambient_light=(0.45, 0.45, 0.45))
        flip = np.array([1, -1, -1], np.float32)   # OpenCV 座標系 -> OpenGL 座標系
        mat = pyrender.MetallicRoughnessMaterial(
            metallicFactor=0.15, roughnessFactor=0.7, alphaMode='OPAQUE',
            baseColorFactor=[*body_color, 1.0], doubleSided=True)
        scene.add(pyrender.Mesh.from_trimesh(trimesh.Trimesh(verts * flip, faces), material=mat))
        if extra is not None:
            ev, ef, ec = extra
            m = trimesh.Trimesh(ev * flip, ef, process=False)
            m.visual.face_colors = np.concatenate(
                [np.clip(ec, 0, 1), np.ones((len(ec), 1), np.float32)], axis=1)
            scene.add(pyrender.Mesh.from_trimesh(m, smooth=False))
        cam = pyrender.IntrinsicsCamera(fx=float(K[0, 0]), fy=float(K[1, 1]),
                                        cx=float(K[0, 2]), cy=float(K[1, 2]),
                                        znear=0.05, zfar=100.0)
        scene.add(cam, pose=np.eye(4))
        for pose in _raymond_light_poses():
            scene.add(pyrender.DirectionalLight(color=np.ones(3), intensity=2.0), pose=pose)
        color, _ = self.renderer.render(scene)
        return np.asarray(color)[..., :3]


def _raymond_light_poses():
    poses = []
    for phi in [0.0, 2 * np.pi / 3, 4 * np.pi / 3]:
        theta = np.pi / 6
        z = np.array([np.sin(theta) * np.cos(phi), np.sin(theta) * np.sin(phi), np.cos(theta)])
        z /= np.linalg.norm(z)
        x = np.array([-z[1], z[0], 0.0])
        x = x / np.linalg.norm(x) if np.linalg.norm(x) > 0 else np.array([1.0, 0.0, 0.0])
        m = np.eye(4)
        m[:3, :3] = np.c_[x, np.cross(z, x), z]
        poses.append(m)
    return poses


# ---- 出力サイズ・カメラ・床の準備 ----
out_h = int(OUT_HEIGHT) // 2 * 2
out_w = int(round(out_h * SEG_W / max(SEG_H, 1))) // 2 * 2
IMSHAPE = (out_h, out_w)

render_verts = MESH_VERTS.copy()
if VIEW_AZIMUTH_DEG:
    center = np.median(render_verts.reshape(-1, 3), axis=0)
    render_verts = rotate_about_y(render_verts, center, VIEW_AZIMUTH_DEG)

if CAMERA_MODE == 'original' and not VIEW_AZIMUTH_DEG:
    K_RENDER = K_ORIG * np.array([[out_w / SEG_W], [out_h / SEG_H], [1.0]], np.float32)
else:
    K_RENDER = fit_camera(render_verts[::max(1, len(render_verts) // 60)], IMSHAPE)

floor = None
if SHOW_FLOOR:
    # 8c が接地から床を推定していればそれを使う（頂点の分位点より安定）
    y_floor = float(globals().get('FLOOR_Y_MM', np.nan)) / 1000.0
    if not np.isfinite(y_floor):
        y_floor = float(np.percentile(render_verts[..., 1], 99.7))
    xz = render_verts.reshape(-1, 3)[:, [0, 2]]
    span = float(max(np.ptp(np.percentile(xz[:, 0], [1, 99])),
                     np.ptp(np.percentile(xz[:, 1], [1, 99]))))
    floor = make_floor_grid([float(np.median(xz[:, 0])), float(np.median(xz[:, 1]))],
                            y_floor, half_size=max(1.2, span * 1.5 + 0.8))

body_rgb = hex2rgb(BODY_COLOR)
bg_rgb = hex2rgb(BG_COLOR)
backend = None
if RENDER_BACKEND == 'pyrender':
    try:
        backend = PyrenderBackend(IMSHAPE, bg_rgb)
        print('pyrender を使います。')
    except Exception as e:
        print('pyrender を初期化できませんでした（', repr(e), '）→ ソフトウェアレンダラを使います。')

# ---- 描画ループ ----
from PIL import Image

SILENT_MP4 = os.path.join(WORK_DIR, 'mannequin_silent.mp4')
src_reader = imageio.get_reader(SEGMENT_MP4) if SIDE_BY_SIDE else None
writer = imageio.get_writer(SILENT_MP4, fps=FPS, codec='libx264', quality=8,
                            macro_block_size=1, pixelformat='yuv420p')
for i in tqdm(range(len(render_verts)), desc='描画中'):
    if backend is not None:
        img = backend.render(render_verts[i], FACES, K_RENDER, IMSHAPE, body_rgb, bg_rgb, floor)
    else:
        img = render_software(render_verts[i], FACES, K_RENDER, IMSHAPE, body_rgb, bg_rgb, floor)
    if src_reader is not None:
        try:
            src = np.asarray(Image.fromarray(src_reader.get_data(i)).resize(
                (out_w, out_h), Image.BILINEAR))
            img = np.concatenate([src, img], axis=1)
        except Exception:
            pass
    writer.append_data(img)
writer.close()
if src_reader is not None:
    src_reader.close()
print('マネキン動画（無音）:', SILENT_MP4, f'({os.path.getsize(SILENT_MP4) / 1024 ** 2:.1f} MB)')

---
## 12. 音声を合成して完成 🎬

切り出しておいた音声を重ねて `mannequin_with_audio.mp4` を書き出し、その場で再生・ダウンロードします。

In [ ]:
#@title 12. 音声付き mp4 を書き出して再生 { display-mode: "form" }
import base64
from IPython.display import HTML, display

FINAL_MP4 = os.path.join(WORK_DIR, 'mannequin_with_audio.mp4')
if AUDIO_PATH:
    run_ffmpeg(['-y', '-loglevel', 'error', '-i', SILENT_MP4, '-i', AUDIO_PATH,
                '-c:v', 'copy', '-c:a', 'aac', '-shortest', FINAL_MP4])
    print('音声を合成しました。')
else:
    run_ffmpeg(['-y', '-loglevel', 'error', '-i', SILENT_MP4, '-c', 'copy', FINAL_MP4])
    print('元動画に音声が無かったため、無音で出力しました。')

size_mb = os.path.getsize(FINAL_MP4) / 1024 ** 2
print('完成:', FINAL_MP4, f'({size_mb:.1f} MB)')

if size_mb < 60:
    b64 = base64.b64encode(open(FINAL_MP4, 'rb').read()).decode()
    display(HTML(f'<video width="480" controls loop>'
                 f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))
else:
    print('（ファイルが大きいのでインライン再生は省略しました）')

if IN_COLAB:
    from google.colab import files
    files.download(FINAL_MP4)
    files.download(MOTION_NPZ)
print('モーションデータ:', MOTION_NPZ)

---
## 13. うまくいかないときは

| 症状 | 対処 |
|---|---|
| `CUDA out of memory` | `BATCH_SIZE` を 1〜2 に、`MAX_HEIGHT` を 480 に下げる |
| GPU が無いというエラー | Colab のランタイムを GPU に変更して最初から実行し直す |
| 人物が検出されない | 区間を人物が大きく写っているところに変える。セル 7 の `DETECTOR_THRESHOLD` を 0.15 程度に下げる |
| 別の人に乗り移る | `PERSON_SELECT` を変える。セル 7 の追跡しきい値 `1.5`（m）を小さくする |
| 体が細かく震える | `NUM_AUG` を 5 に上げる（推定ノイズを発生源で減らす）。`ROOT_CUTOFF_HZ` を 2.0 に下げる |
| 動きがぼやける・キレがなくなる | `SMOOTH_CUTOFF_HZ` を 8〜10 に上げる。`ROOT_CUTOFF_HZ` も上げる |
| 足が地面を滑る | `FOOT_LOCK=True` のまま `ROOT_MOTION='locked'` にする。8c のグラフで接地区間（灰色）が出ていなければ `CONTACT_HEIGHT_MM` / `CONTACT_SPEED_MMPS` を大きくする |
| 足が地面にめり込む / 浮く | `CONTACT_HEIGHT_MM` を調整。接地が全く検出されていないと垂直補正も効きません |
| その場で踊ってほしくない（移動を残したい） | `ROOT_MOTION='full'` |
| GPU が余っている | `BATCH_SIZE` → `NUM_AUG` の順に上げる（クロップ枚数 = フレーム数 × 人数 × NUM_AUG） |
| マネキンが小さい / 見切れる | `CAMERA_MODE='fit'` にする。`fit_camera` の `margin` を調整する |
| 全身が入っていない動画 | NLF は部分的な人体でも推定しますが、全身が写っている区間のほうが安定します |
| 処理が遅い | `TARGET_FPS` を 15 に、`MAX_HEIGHT` を 480 に下げる |

### 出力ファイル

| ファイル | 内容 |
|---|---|
| `nlf_mannequin/motion.npz` | 抽出したモーションデータ（`pose` (T,24,3) 回転ベクトル、`betas`、`trans`、`joints3d`、`vertices3d`、`ground_offset_mm`、`fps` など）。セル 8 / 8b / 8c のどれを実行しても最新の状態で保存し直されます |
| `nlf_mannequin/mannequin_silent.mp4` | マネキン動画（無音） |
| `nlf_mannequin/mannequin_with_audio.mp4` | **最終出力**（音声付き） |

### 参考

* NLF: <https://github.com/isarandi/nlf> — [NeurIPS 2024 論文](https://arxiv.org/abs/2407.07532)
* NLF v0.3.2 リリース: <https://github.com/isarandi/nlf/releases/tag/v0.3.2>
* SMPLFitter: <https://github.com/isarandi/smplfitter>

```
@article{sarandi2024nlf,
    title   = {Neural Localizer Fields for Continuous 3D Human Pose and Shape Estimation},
    author  = {S\'ar\'andi, Istv\'an and Pons-Moll, Gerard},
    journal = {Advances in Neural Information Processing Systems (NeurIPS)},
    year    = {2024}
}
```